# ============================================================
# TURKQUISH — EXPORT 5-CLASS DEPLOYMENT ARTIFACTS
# ============================================================
# Run this AFTER your 5-class model is trained.
# This cell does NOT retrain the model.
#
# It exports:
#   artifacts/
#     model.joblib
#     preprocessing_pipeline.joblib
#     feature_schema.json
#     label_encoder.json
#     threshold.json
#     frozen_token_graph.pkl
#     explanation_templates.json
#     model_card.json
#     turkquish_5class_artifacts.zip
#
# IMPORTANT:
# - Your fitted model must already exist in notebook memory.
# - The real frozen graph/projector should already exist in notebook memory.
# - If graph object is missing, this cell can optionally build fallback frozen
#   token statistics from train_df, but the best option is exporting the exact
#   graph/projector used during training.
# ============================================================


In [ ]:

from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
from collections import Counter, defaultdict
import json
import pickle
import shutil
import re
import inspect
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline


# ============================================================
# 0. USER SETTINGS — CHANGE ONLY THIS PART IF NEEDED
# ============================================================

# Output folder
ARTIFACT_DIR = Path("/content/artifacts_5class")

# Optional manual variable names.
# Leave as None for auto-detection.
# Example:
# MANUAL_MODEL_VAR_NAME = "histgb_5class"
# MANUAL_X_TRAIN_VAR_NAME = "X_train_5class"
# MANUAL_GRAPH_VAR_NAME = "token_graph_train"
MANUAL_MODEL_VAR_NAME = None
MANUAL_PREPROCESSOR_VAR_NAME = None
MANUAL_GRAPH_VAR_NAME = None
MANUAL_FEATURE_NAMES_VAR_NAME = None
MANUAL_X_TRAIN_VAR_NAME = None
MANUAL_TRAIN_DF_VAR_NAME = None

# Used only if graph object is missing and fallback graph stats must be built
URL_COL = "url"
LABEL_COL = "class_enc"

# Deployment checks
STRICT_5CLASS = True
STRICT_135_FEATURES = True

# Best: export real graph/projector from notebook memory.
# Fallback: build frozen token statistics from training URLs.
# This is NOT retraining, but it may not exactly match your original graph features.
ALLOW_FALLBACK_GRAPH_STATS = True

# Optional threshold tuning if validation data exists.
# Leave None for auto-detection/default 0.50.
MANUAL_Y_VAL_VAR_NAME = None
MANUAL_X_VAL_VAR_NAME = None
MANUAL_PROBA_VAL_VAR_NAME = None


# ============================================================
# 1. HELPERS
# ============================================================

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def utc_now():
    return datetime.utcnow().isoformat() + "Z"


def json_default(obj):
    """Make numpy/pandas/sklearn metadata JSON serializable."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp,)):
        return obj.isoformat()
    return str(obj)


def get_global(name):
    if name is None:
        return None
    if name not in globals():
        raise NameError(f"Manual variable name '{name}' not found in notebook globals().")
    return globals()[name]


def is_fitted_model(obj):
    return (
        hasattr(obj, "predict")
        and hasattr(obj, "predict_proba")
        and hasattr(obj, "classes_")
    )


def safe_len_classes(obj):
    try:
        return len(getattr(obj, "classes_", []))
    except Exception:
        return 0


def print_box(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


# ============================================================
# 2. FIND THE 5-CLASS MODEL
# ============================================================

print_box("STEP 1 — Finding fitted 5-class model")

model_5class = get_global(MANUAL_MODEL_VAR_NAME)

if model_5class is None:
    candidates = []

    for name, obj in list(globals().items()):
        try:
            if is_fitted_model(obj) and safe_len_classes(obj) == 5:
                candidates.append((name, type(obj), getattr(obj, "classes_", None)))
        except Exception:
            pass

    print("5-class model candidates found:")
    for name, typ, classes in candidates:
        print(f"  - {name}: {typ}, classes={classes}")

    if len(candidates) == 0:
        raise RuntimeError(
            "No fitted 5-class model found in notebook memory.\n"
            "Run the 5-class training cell first, or set:\n"
            "MANUAL_MODEL_VAR_NAME = 'your_model_variable_name'"
        )

    if len(candidates) > 1:
        preferred_keywords = [
            "histgb", "hist_gradient", "best", "final", "model_5", "five", "multiclass"
        ]
        scored = []
        for item in candidates:
            name = item[0].lower()
            score = sum(k in name for k in preferred_keywords)
            scored.append((score, item))

        scored = sorted(scored, key=lambda x: x[0], reverse=True)
        if scored[0][0] == 0:
            raise RuntimeError(
                "Multiple 5-class model candidates found. Please set:\n"
                "MANUAL_MODEL_VAR_NAME = 'correct_model_name'\n"
                f"Candidates: {[c[0] for c in candidates]}"
            )

        model_name, model_type, model_classes = scored[0][1]
        model_5class = globals()[model_name]
        print(f"\nAuto-selected model: {model_name}")
    else:
        model_name, model_type, model_classes = candidates[0]
        model_5class = globals()[model_name]
        print(f"\nAuto-selected model: {model_name}")
else:
    model_name = MANUAL_MODEL_VAR_NAME
    print(f"Using manual model: {model_name}")

print("Model type:", type(model_5class))
print("Model classes:", getattr(model_5class, "classes_", None))

if STRICT_5CLASS:
    assert hasattr(model_5class, "classes_"), "Model has no classes_ attribute."
    assert len(model_5class.classes_) == 5, (
        f"Expected 5 classes, got {len(model_5class.classes_)}: {model_5class.classes_}"
    )


# ============================================================
# 3. FIND FEATURE NAMES / FEATURE SCHEMA
# ============================================================

print_box("STEP 2 — Finding feature names")

feature_names = None

manual_feature_names = get_global(MANUAL_FEATURE_NAMES_VAR_NAME)
if manual_feature_names is not None:
    feature_names = list(manual_feature_names)
    print(f"Using manual feature names: {MANUAL_FEATURE_NAMES_VAR_NAME}")

# Try model.feature_names_in_
if feature_names is None and hasattr(model_5class, "feature_names_in_"):
    feature_names = list(model_5class.feature_names_in_)
    print("Loaded feature names from model.feature_names_in_")

# Try pipeline first step / final estimator
if feature_names is None and isinstance(model_5class, Pipeline):
    try:
        if hasattr(model_5class, "feature_names_in_"):
            feature_names = list(model_5class.feature_names_in_)
            print("Loaded feature names from pipeline.feature_names_in_")
    except Exception:
        pass

    if feature_names is None:
        try:
            final_est = model_5class.steps[-1][1]
            if hasattr(final_est, "feature_names_in_"):
                feature_names = list(final_est.feature_names_in_)
                print("Loaded feature names from final estimator.feature_names_in_")
        except Exception:
            pass

# Try manual X_train
X_train_obj = get_global(MANUAL_X_TRAIN_VAR_NAME)
if feature_names is None and X_train_obj is not None:
    if hasattr(X_train_obj, "columns"):
        feature_names = list(X_train_obj.columns)
        print(f"Loaded feature names from manual X_train: {MANUAL_X_TRAIN_VAR_NAME}")

# Auto-detect X_train dataframe
if feature_names is None:
    x_candidates = []
    for name, obj in list(globals().items()):
        try:
            if isinstance(obj, pd.DataFrame):
                lname = name.lower()
                if (
                    lname.startswith("x_train")
                    or "x_train" in lname
                    or "xtr" in lname
                    or "features_train" in lname
                    or "train_features" in lname
                ):
                    # likely feature matrix: many numeric columns
                    numeric_ratio = obj.select_dtypes(include=[np.number]).shape[1] / max(1, obj.shape[1])
                    if obj.shape[1] >= 20 and numeric_ratio > 0.80:
                        x_candidates.append((name, obj.shape, numeric_ratio))
        except Exception:
            pass

    print("X_train candidates:")
    for c in x_candidates[:20]:
        print("  -", c)

    if len(x_candidates) == 1:
        x_name = x_candidates[0][0]
        X_train_obj = globals()[x_name]
        feature_names = list(X_train_obj.columns)
        print(f"Loaded feature names from auto-detected X_train: {x_name}")
    elif len(x_candidates) > 1:
        raise RuntimeError(
            "Multiple X_train candidates found. Please set:\n"
            "MANUAL_X_TRAIN_VAR_NAME = 'correct_X_train_variable_name'\n"
            f"Candidates: {[c[0] for c in x_candidates]}"
        )

# Try common feature list variables
if feature_names is None:
    list_candidates = []
    for name, obj in list(globals().items()):
        try:
            if isinstance(obj, (list, tuple, pd.Index, np.ndarray)):
                lname = name.lower()
                if (
                    "feature" in lname
                    and len(obj) >= 20
                    and all(isinstance(x, str) for x in list(obj)[:10])
                ):
                    list_candidates.append((name, len(obj)))
        except Exception:
            pass

    print("Feature-list candidates:")
    for c in list_candidates[:20]:
        print("  -", c)

    if len(list_candidates) == 1:
        feature_names = list(globals()[list_candidates[0][0]])
        print(f"Loaded feature names from feature-list variable: {list_candidates[0][0]}")
    elif len(list_candidates) > 1:
        preferred = [x for x in list_candidates if any(k in x[0].lower() for k in ["final", "selected", "used", "schema"])]
        if len(preferred) == 1:
            feature_names = list(globals()[preferred[0][0]])
            print(f"Loaded feature names from preferred feature-list variable: {preferred[0][0]}")
        else:
            raise RuntimeError(
                "Multiple feature-list candidates found. Please set:\n"
                "MANUAL_FEATURE_NAMES_VAR_NAME = 'correct_feature_list_variable_name'\n"
                f"Candidates: {[c[0] for c in list_candidates]}"
            )

if feature_names is None:
    raise RuntimeError(
        "Could not determine feature names.\n"
        "Set MANUAL_X_TRAIN_VAR_NAME or MANUAL_FEATURE_NAMES_VAR_NAME."
    )

feature_names = [str(x) for x in feature_names]
print("Number of features:", len(feature_names))

if STRICT_135_FEATURES:
    assert len(feature_names) == 135, (
        f"Expected 135 deployment features, got {len(feature_names)}.\n"
        "You may be exporting the wrong model/regime, or artefact columns were not removed.\n"
        "If you intentionally want to export a different model, set STRICT_135_FEATURES = False."
    )


# ============================================================
# 4. FIND / EXPORT PREPROCESSING PIPELINE
# ============================================================

print_box("STEP 3 — Finding preprocessing pipeline")

preprocessing_pipeline = get_global(MANUAL_PREPROCESSOR_VAR_NAME)

if isinstance(model_5class, Pipeline):
    # Model already contains preprocessing inside the pipeline.
    # Save model as full pipeline and use identity preprocessor in backend.
    preprocessing_pipeline = FunctionTransformer(validate=False)
    print("Model is already a sklearn Pipeline.")
    print("Saving full pipeline as model.joblib and identity preprocessing_pipeline.joblib.")
else:
    if preprocessing_pipeline is not None:
        print(f"Using manual preprocessing pipeline: {MANUAL_PREPROCESSOR_VAR_NAME}")
    else:
        prep_candidates = []
        for name, obj in list(globals().items()):
            try:
                if obj is model_5class:
                    continue

                lname = name.lower()
                has_transform = hasattr(obj, "transform")
                likely_name = any(
                    k in lname for k in [
                        "preprocess", "preprocessor", "pipeline", "imputer",
                        "scaler", "transformer", "processor"
                    ]
                )
                fitted_attrs = any(
                    hasattr(obj, attr)
                    for attr in [
                        "statistics_", "mean_", "scale_", "n_features_in_",
                        "feature_names_in_", "transformers_", "steps"
                    ]
                )

                if has_transform and likely_name and fitted_attrs:
                    prep_candidates.append((name, type(obj)))
            except Exception:
                pass

        print("Preprocessor candidates:")
        for c in prep_candidates:
            print("  -", c)

        if len(prep_candidates) == 1:
            prep_name = prep_candidates[0][0]
            preprocessing_pipeline = globals()[prep_name]
            print(f"Auto-selected preprocessing pipeline: {prep_name}")
        elif len(prep_candidates) > 1:
            raise RuntimeError(
                "Multiple preprocessing candidates found. Please set:\n"
                "MANUAL_PREPROCESSOR_VAR_NAME = 'correct_preprocessor_variable_name'\n"
                f"Candidates: {[c[0] for c in prep_candidates]}"
            )
        else:
            preprocessing_pipeline = FunctionTransformer(validate=False)
            print("No separate preprocessing object found.")
            print("Using identity preprocessing pipeline.")

print("Preprocessor type:", type(preprocessing_pipeline))


# ============================================================
# 5. SAVE MODEL AND PREPROCESSOR
# ============================================================

print_box("STEP 4 — Saving model and preprocessing artifacts")

model_path = ARTIFACT_DIR / "model.joblib"
preprocessor_path = ARTIFACT_DIR / "preprocessing_pipeline.joblib"

joblib.dump(model_5class, model_path)
joblib.dump(preprocessing_pipeline, preprocessor_path)

print("Saved:", model_path)
print("Saved:", preprocessor_path)


# ============================================================
# 6. SAVE FEATURE SCHEMA
# ============================================================

print_box("STEP 5 — Saving feature_schema.json")

def infer_feature_group(feature_name: str):
    n = feature_name.lower()

    graph_markers = [
        "graph", "cluster", "component", "degree", "centrality",
        "pagerank", "token_reuse", "campaign", "family", "cooccurrence",
        "co_occur", "neighbor"
    ]
    turkish_markers = [
        "turkish", "tr_", "vowel", "harmony", "suffix", "morph",
        "bigram", "translit", "ç", "ğ", "ı", "ö", "ş", "ü"
    ]
    brand_markers = [
        "brand", "typo", "squat", "imperson", "homoglyph",
        "punycode", "obfus", "redirect", "tld_mismatch"
    ]
    keyword_markers = [
        "phish", "malware", "scam", "keyword", "lexicon", "threat_vocab"
    ]

    if any(x in n for x in graph_markers):
        return "graph_infrastructure"
    if any(x in n for x in turkish_markers):
        return "turkish_linguistic"
    if any(x in n for x in brand_markers):
        return "adversarial_brand"
    if any(x in n for x in keyword_markers):
        return "lexical_keyword"
    return "lexical_structural"


feature_groups = {}
for feat in feature_names:
    group = infer_feature_group(feat)
    feature_groups.setdefault(group, []).append(feat)

feature_schema = {
    "schema_name": "turkquish_5class_feature_schema",
    "schema_version": "tumc-135-5class-v1" if len(feature_names) == 135 else f"tumc-{len(feature_names)}-5class-v1",
    "created_at": utc_now(),
    "n_features": len(feature_names),
    "features": feature_names,
    "feature_groups": feature_groups,
    "target_classes": [
        "benign",
        "phishing",
        "malware",
        "scam",
        "other_malicious"
    ],
    "url_only": True,
    "forbidden_runtime_features": [
        "DNS",
        "WHOIS",
        "HTML_fetch",
        "webpage_screenshot",
        "third_party_reputation_query",
        "live_page_retrieval"
    ],
    "notes": (
        "Feature order must exactly match the order used during 5-class model training. "
        "Backend must construct a pandas DataFrame with these columns in this exact order."
    )
}

with open(ARTIFACT_DIR / "feature_schema.json", "w", encoding="utf-8") as f:
    json.dump(feature_schema, f, indent=2, ensure_ascii=False, default=json_default)

print("Saved:", ARTIFACT_DIR / "feature_schema.json")


# ============================================================
# 7. SAVE LABEL ENCODER / CLASS ORDER
# ============================================================

print_box("STEP 6 — Saving label_encoder.json")

canonical_id_to_label = {
    0: "benign",
    1: "phishing",
    2: "malware",
    3: "scam",
    4: "other_malicious"
}

model_classes = list(getattr(model_5class, "classes_", []))
model_classes_json = [json_default(c) for c in model_classes]

# If model classes are integer-coded 0..4
if set([int(c) for c in model_classes]) == {0, 1, 2, 3, 4}:
    model_class_order_labels = [canonical_id_to_label[int(c)] for c in model_classes]
else:
    # If model uses string labels, preserve exact model order
    model_class_order_labels = [str(c) for c in model_classes]

label_encoder = {
    "label_type": "fixed_5class_mapping",
    "created_at": utc_now(),
    "classes": [
        "benign",
        "phishing",
        "malware",
        "scam",
        "other_malicious"
    ],
    "id_to_label": {str(k): v for k, v in canonical_id_to_label.items()},
    "label_to_id": {v: k for k, v in canonical_id_to_label.items()},
    "model_classes_raw_order": model_classes_json,
    "model_classes_label_order": model_class_order_labels,
    "probability_order_note": (
        "predict_proba output columns follow model.classes_. "
        "Use model_classes_label_order to map probabilities correctly."
    )
}

with open(ARTIFACT_DIR / "label_encoder.json", "w", encoding="utf-8") as f:
    json.dump(label_encoder, f, indent=2, ensure_ascii=False, default=json_default)

print("Saved:", ARTIFACT_DIR / "label_encoder.json")
print("Probability label order:", model_class_order_labels)


# ============================================================
# 8. SAVE THRESHOLD
# ============================================================

print_box("STEP 7 — Saving threshold.json")

def compute_threshold_if_possible():
    """
    Compute binary malicious threshold for riskScore = 1 - P(benign)
    using validation data if available. Otherwise return default 0.50.
    """
    from sklearn.metrics import f1_score

    y_val = get_global(MANUAL_Y_VAL_VAR_NAME)
    X_val = get_global(MANUAL_X_VAL_VAR_NAME)
    proba_val = get_global(MANUAL_PROBA_VAL_VAR_NAME)

    # Auto-detect validation probabilities first
    if proba_val is None:
        for name, obj in list(globals().items()):
            try:
                lname = name.lower()
                if (
                    isinstance(obj, np.ndarray)
                    and obj.ndim == 2
                    and obj.shape[1] == 5
                    and ("proba" in lname or "prob" in lname)
                    and ("val" in lname or "valid" in lname or "test" in lname)
                ):
                    proba_val = obj
                    print(f"Auto-detected validation probabilities: {name}")
                    break
            except Exception:
                pass

    # Auto-detect y validation
    if y_val is None:
        for name, obj in list(globals().items()):
            try:
                lname = name.lower()
                if (
                    ("y_val" in lname or "y_valid" in lname or "y_test" in lname)
                    and isinstance(obj, (pd.Series, np.ndarray, list))
                    and len(obj) > 10
                ):
                    arr = np.asarray(obj)
                    unique = set(pd.Series(arr).dropna().astype(int).unique().tolist())
                    if unique.issubset({0, 1, 2, 3, 4}) and len(unique) >= 2:
                        y_val = obj
                        print(f"Auto-detected y validation: {name}")
                        break
            except Exception:
                pass

    # Auto-detect X validation only if probabilities absent
    if proba_val is None and X_val is None:
        for name, obj in list(globals().items()):
            try:
                lname = name.lower()
                if (
                    isinstance(obj, pd.DataFrame)
                    and ("x_val" in lname or "x_valid" in lname or "x_test" in lname)
                    and set(feature_names).issubset(set(obj.columns))
                ):
                    X_val = obj[feature_names]
                    print(f"Auto-detected X validation: {name}")
                    break
            except Exception:
                pass

    # Compute probabilities if X_val available
    if proba_val is None and X_val is not None:
        try:
            Xv = X_val[feature_names] if hasattr(X_val, "columns") else X_val
            Xv_proc = preprocessing_pipeline.transform(Xv)
            proba_val = model_5class.predict_proba(Xv_proc)
            print("Computed validation probabilities from X_val.")
        except Exception as e:
            warnings.warn(f"Could not compute validation probabilities: {e}")

    if y_val is None or proba_val is None:
        print("Validation data/probabilities not found. Using default threshold = 0.50")
        return 0.50, None

    y_arr = np.asarray(y_val).astype(int)
    proba_arr = np.asarray(proba_val)

    if proba_arr.ndim != 2 or proba_arr.shape[1] != 5:
        print("Validation probabilities are not shape (n,5). Using default threshold = 0.50")
        return 0.50, None

    # Find benign column according to model.classes_
    classes_as_int = [int(c) for c in model_5class.classes_]
    benign_col = classes_as_int.index(0)

    risk_scores = 1.0 - proba_arr[:, benign_col]
    y_binary = (y_arr != 0).astype(int)

    best_tau = 0.50
    best_f1 = -1.0

    for tau in np.linspace(0.05, 0.95, 181):
        pred_binary = (risk_scores >= tau).astype(int)
        f1 = f1_score(y_binary, pred_binary)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_tau = float(tau)

    print(f"Best validation threshold: {best_tau:.3f}")
    print(f"Best binary malicious-vs-benign F1: {best_f1:.4f}")

    return best_tau, best_f1


tau, threshold_validation_f1 = compute_threshold_if_possible()

threshold = {
    "threshold_type": "malicious_risk_threshold",
    "created_at": utc_now(),
    "risk_score_formula": "1 - P(benign)",
    "benign_class_id": 0,
    "threshold": float(tau),
    "validation_binary_f1": threshold_validation_f1,
    "decision_rule": {
        "benign": "risk_score < threshold",
        "suspicious_or_malicious": "risk_score >= threshold"
    },
    "recommended_action_rules": [
        {"min": 0.00, "max": 0.30, "action": "proceed", "riskLevel": "low"},
        {"min": 0.30, "max": 0.60, "action": "caution", "riskLevel": "medium"},
        {"min": 0.60, "max": 0.85, "action": "block", "riskLevel": "high"},
        {"min": 0.85, "max": 1.01, "action": "report", "riskLevel": "critical"}
    ]
}

with open(ARTIFACT_DIR / "threshold.json", "w", encoding="utf-8") as f:
    json.dump(threshold, f, indent=2, ensure_ascii=False, default=json_default)

print("Saved:", ARTIFACT_DIR / "threshold.json")


# ============================================================
# 9. SAVE FROZEN TOKEN GRAPH / PROJECTOR
# ============================================================

print_box("STEP 8 — Saving frozen_token_graph.pkl")

def looks_like_graph_or_projector(obj):
    """Detect graph/projector/transformer objects without being too strict."""
    if obj is None:
        return False

    if inspect.ismodule(obj) or inspect.isfunction(obj) or inspect.isclass(obj):
        return False

    type_name = str(type(obj)).lower()

    # NetworkX-like graph
    if hasattr(obj, "number_of_nodes") and hasattr(obj, "number_of_edges"):
        return True

    # Projector / transformer style
    if hasattr(obj, "transform") or hasattr(obj, "project") or hasattr(obj, "fit_transform"):
        if "graph" in type_name or "project" in type_name or "token" in type_name:
            return True

    # Dict bundle
    if isinstance(obj, dict):
        keys = set(str(k).lower() for k in obj.keys())
        important = {
            "graph", "token_counts", "token_to_domains", "domain_to_component",
            "component_sizes", "degree", "centrality", "token_label_counts",
            "graph_version"
        }
        if len(keys.intersection(important)) >= 2:
            return True

    return False


def simple_url_tokens(url: str):
    """URL-only tokenizer for fallback frozen graph statistics."""
    url = str(url).strip().lower()
    if not url:
        return "", []

    if "://" not in url:
        parsed = urlparse("http://" + url)
    else:
        parsed = urlparse(url)

    host = parsed.netloc.lower()
    path = parsed.path.lower()
    query = parsed.query.lower()

    raw = host + " " + path + " " + query
    tokens = re.split(r"[^a-z0-9çğıöşü]+", raw)
    tokens = [t for t in tokens if 3 <= len(t) <= 80]

    return host, tokens


def find_train_df_for_graph():
    manual = get_global(MANUAL_TRAIN_DF_VAR_NAME)
    if manual is not None:
        return MANUAL_TRAIN_DF_VAR_NAME, manual

    candidates = []
    for name, obj in list(globals().items()):
        try:
            if not isinstance(obj, pd.DataFrame):
                continue

            cols = set(obj.columns)
            lower_cols = {str(c).lower() for c in obj.columns}

            has_url = URL_COL in cols or "url" in lower_cols
            has_label = LABEL_COL in cols or "class_enc" in lower_cols or "label" in lower_cols or "class" in lower_cols

            lname = name.lower()
            likely_name = "train" in lname or "df" in lname or "tumc" in lname

            if has_url and has_label and likely_name:
                candidates.append((name, obj.shape, list(obj.columns)[:10]))
        except Exception:
            pass

    print("Training dataframe candidates for graph fallback:")
    for c in candidates[:20]:
        print("  -", c)

    if len(candidates) == 1:
        return candidates[0][0], globals()[candidates[0][0]]

    if len(candidates) > 1:
        preferred = [
            c for c in candidates
            if any(k in c[0].lower() for k in ["train", "train_df", "df_train", "train_fold"])
        ]
        if len(preferred) == 1:
            return preferred[0][0], globals()[preferred[0][0]]

        raise RuntimeError(
            "Multiple possible training dataframes found for graph fallback.\n"
            "Set MANUAL_TRAIN_DF_VAR_NAME = 'correct_train_df_variable_name'\n"
            f"Candidates: {[c[0] for c in candidates]}"
        )

    return None, None


def build_fallback_frozen_token_graph_stats(train_df, url_col=URL_COL, label_col=LABEL_COL):
    """
    Fallback frozen graph-stat bundle.
    This is NOT model retraining.
    It builds deployment-time token statistics from training URLs only.

    Best practice:
    Use this only if your original graph/projector object is unavailable.
    For journal-consistent deployment, export the exact graph/projector used
    during training.
    """
    df = train_df.copy()

    # Resolve URL column
    if url_col not in df.columns:
        url_candidates = [c for c in df.columns if str(c).lower() == "url"]
        if not url_candidates:
            raise ValueError(f"URL column '{url_col}' not found.")
        url_col = url_candidates[0]

    # Resolve label column
    if label_col not in df.columns:
        for cand in ["class_enc", "label", "class", "y"]:
            if cand in df.columns:
                label_col = cand
                break
        else:
            raise ValueError(f"Label column '{label_col}' not found.")

    token_doc_freq = Counter()
    token_label_counts = defaultdict(lambda: Counter())
    domain_doc_freq = Counter()
    domain_label_counts = defaultdict(lambda: Counter())
    domain_token_counts = defaultdict(Counter)

    n_rows = len(df)
    print(f"Building fallback frozen token statistics from {n_rows:,} training URLs...")

    for i, row in enumerate(df[[url_col, label_col]].itertuples(index=False, name=None), start=1):
        url, label = row
        try:
            label = int(label)
        except Exception:
            # map string labels if needed
            label_map = {
                "benign": 0,
                "phishing": 1,
                "malware": 2,
                "scam": 3,
                "other_malicious": 4,
                "other-malicious": 4,
                "other": 4
            }
            label = label_map.get(str(label).lower(), 4)

        domain, tokens = simple_url_tokens(url)
        unique_tokens = sorted(set(tokens))

        if domain:
            domain_doc_freq[domain] += 1
            domain_label_counts[domain][label] += 1

        for tok in unique_tokens:
            token_doc_freq[tok] += 1
            token_label_counts[tok][label] += 1
            if domain:
                domain_token_counts[domain][tok] += 1

        if i % 200000 == 0:
            print(f"  processed {i:,}/{n_rows:,}")

    # Convert counters to normal JSON/pickle-friendly dicts
    token_label_counts_dict = {
        tok: {str(k): int(v) for k, v in cnt.items()}
        for tok, cnt in token_label_counts.items()
    }
    domain_label_counts_dict = {
        dom: {str(k): int(v) for k, v in cnt.items()}
        for dom, cnt in domain_label_counts.items()
    }

    bundle = {
        "graph_version": "tumc-token-graph-fallback-stats-5class-v1",
        "created_at": utc_now(),
        "protocol": "fallback_training_token_statistics",
        "important_warning": (
            "This fallback bundle was built from training URLs only and is safe from live-data leakage, "
            "but it may not exactly match the original graph features used during model training. "
            "For final journal deployment, export the exact graph/projector object used during training."
        ),
        "url_col": str(url_col),
        "label_col": str(label_col),
        "classes": canonical_id_to_label,
        "n_training_rows": int(n_rows),
        "token_doc_freq": dict(token_doc_freq),
        "token_label_counts": token_label_counts_dict,
        "domain_doc_freq": dict(domain_doc_freq),
        "domain_label_counts": domain_label_counts_dict,
        "projection_rule": (
            "At inference time, tokenize the new URL and look up tokens/domains in this frozen bundle. "
            "Do not update this bundle with live traffic."
        )
    }

    return bundle


frozen_graph_object = get_global(MANUAL_GRAPH_VAR_NAME)
graph_source = None

if frozen_graph_object is not None:
    graph_source = MANUAL_GRAPH_VAR_NAME
    print(f"Using manual graph/projector object: {graph_source}")
else:
    graph_candidates = []

    preferred_names = [
        "frozen_token_graph",
        "token_graph",
        "token_graph_train",
        "G_train",
        "train_graph",
        "graph_projector",
        "inductive_graph",
        "token_cooccurrence_graph",
        "cooccurrence_graph",
        "graph_features",
        "graph_transformer"
    ]

    # First check preferred names
    for name in preferred_names:
        if name in globals():
            obj = globals()[name]
            if looks_like_graph_or_projector(obj):
                graph_candidates.append((name, type(obj)))

    # Then scan all globals
    if not graph_candidates:
        for name, obj in list(globals().items()):
            try:
                lname = name.lower()
                if (
                    ("graph" in lname or "projector" in lname or "cooc" in lname)
                    and looks_like_graph_or_projector(obj)
                ):
                    graph_candidates.append((name, type(obj)))
            except Exception:
                pass

    print("Graph/projector candidates:")
    for c in graph_candidates[:30]:
        print("  -", c)

    if len(graph_candidates) == 1:
        graph_source = graph_candidates[0][0]
        frozen_graph_object = globals()[graph_source]
        print(f"Auto-selected graph/projector: {graph_source}")
    elif len(graph_candidates) > 1:
        raise RuntimeError(
            "Multiple graph/projector candidates found. Please set:\n"
            "MANUAL_GRAPH_VAR_NAME = 'correct_graph_variable_name'\n"
            f"Candidates: {[c[0] for c in graph_candidates]}"
        )

# If no graph/projector found, build fallback stats from train_df
if frozen_graph_object is None:
    if not ALLOW_FALLBACK_GRAPH_STATS:
        raise RuntimeError(
            "No frozen graph/projector found.\n"
            "Set MANUAL_GRAPH_VAR_NAME to the exact graph/projector variable used during training."
        )

    train_df_name, train_df_obj = find_train_df_for_graph()
    if train_df_obj is None:
        raise RuntimeError(
            "No graph/projector object found and no training dataframe found for fallback stats.\n"
            "Recommended: rerun your graph-feature construction cell, then set:\n"
            "MANUAL_GRAPH_VAR_NAME = 'your_graph_or_projector_variable_name'"
        )

    print(f"Building fallback frozen token graph stats from: {train_df_name}")
    frozen_graph_object = build_fallback_frozen_token_graph_stats(
        train_df_obj,
        url_col=URL_COL,
        label_col=LABEL_COL
    )
    graph_source = f"fallback_stats_from_{train_df_name}"

# Bundle graph/projector with metadata
frozen_graph_bundle = {
    "artifact_type": "frozen_token_graph_or_projector",
    "created_at": utc_now(),
    "source_variable": graph_source,
    "url_only": True,
    "deployment_protocol": (
        "Frozen training-only graph/projector. New QR-scanned URLs must be projected "
        "onto this object without updating it with live traffic."
    ),
    "object_type": str(type(frozen_graph_object)),
    "object": frozen_graph_object
}

# Add graph stats if available
try:
    if hasattr(frozen_graph_object, "number_of_nodes"):
        frozen_graph_bundle["n_nodes"] = int(frozen_graph_object.number_of_nodes())
    if hasattr(frozen_graph_object, "number_of_edges"):
        frozen_graph_bundle["n_edges"] = int(frozen_graph_object.number_of_edges())
except Exception:
    pass

with open(ARTIFACT_DIR / "frozen_token_graph.pkl", "wb") as f:
    pickle.dump(frozen_graph_bundle, f)

print("Saved:", ARTIFACT_DIR / "frozen_token_graph.pkl")
print("Graph source:", graph_source)
print("Graph object type:", type(frozen_graph_object))


# ============================================================
# 10. SAVE EXPLANATION TEMPLATES
# ============================================================

print_box("STEP 9 — Saving explanation_templates.json")

explanation_templates = {
    "version": "turkquish-explanations-5class-v1",
    "created_at": utc_now(),
    "languages": ["en", "tr"],
    "class_templates": {
        "benign": {
            "en": "The URL appears structurally consistent with benign URLs in the training corpus.",
            "tr": "URL, eğitim verisindeki güvenli URL yapılarıyla uyumlu görünüyor."
        },
        "phishing": {
            "en": "The URL contains credential, login, verification, account, or brand-impersonation signals.",
            "tr": "URL; giriş, doğrulama, hesap veya marka taklidiyle ilişkili sinyaller içeriyor."
        },
        "malware": {
            "en": "The URL contains download, executable, payload, update, or malware-delivery indicators.",
            "tr": "URL; indirme, çalıştırılabilir dosya, zararlı yük, güncelleme veya kötü amaçlı yazılım dağıtım göstergeleri içeriyor."
        },
        "scam": {
            "en": "The URL contains prize, investment, gambling, fake-offer, or social-engineering scam indicators.",
            "tr": "URL; ödül, yatırım, bahis, sahte teklif veya sosyal mühendislik dolandırıcılığı göstergeleri içeriyor."
        },
        "other_malicious": {
            "en": "The URL shows suspicious infrastructure or campaign-level patterns that do not cleanly fit phishing, malware, or scam.",
            "tr": "URL; kimlik avı, kötü amaçlı yazılım veya dolandırıcılık sınıflarına tam uymayan şüpheli altyapı veya kampanya örüntüleri gösteriyor."
        }
    },
    "feature_group_templates": {
        "lexical_structural": {
            "en": "Lexical and structural URL patterns contributed to the decision.",
            "tr": "URL'nin sözcüksel ve yapısal özellikleri karara katkı sağladı."
        },
        "lexical_keyword": {
            "en": "Threat-related keyword patterns contributed to the decision.",
            "tr": "Tehdit ile ilişkili anahtar kelime örüntüleri karara katkı sağladı."
        },
        "turkish_linguistic": {
            "en": "Turkish linguistic or localized threat features contributed to the decision.",
            "tr": "Türkçe dilsel özellikler veya yerelleştirilmiş tehdit özellikleri karara katkı sağladı."
        },
        "adversarial_brand": {
            "en": "Brand impersonation, typosquatting, or adversarial naming signals contributed to the decision.",
            "tr": "Marka taklidi, yazım benzerliği veya saldırgan adlandırma sinyalleri karara katkı sağladı."
        },
        "graph_infrastructure": {
            "en": "Frozen graph-based campaign infrastructure features contributed to the decision.",
            "tr": "Dondurulmuş grafik tabanlı kampanya altyapısı özellikleri karara katkı sağladı."
        }
    },
    "risk_templates": {
        "low": {
            "en": "Low estimated risk. Still verify the destination before entering sensitive information.",
            "tr": "Tahmini risk düşük. Yine de hassas bilgi girmeden önce hedefi doğrulayın."
        },
        "medium": {
            "en": "Medium estimated risk. Proceed only if you trust the source of the QR code.",
            "tr": "Tahmini risk orta. Yalnızca QR kodunun kaynağına güveniyorsanız devam edin."
        },
        "high": {
            "en": "High estimated risk. Opening this URL is not recommended.",
            "tr": "Tahmini risk yüksek. Bu URL'nin açılması önerilmez."
        },
        "critical": {
            "en": "Critical estimated risk. Block or report this URL.",
            "tr": "Tahmini risk kritik. Bu URL'yi engelleyin veya bildirin."
        }
    }
}

with open(ARTIFACT_DIR / "explanation_templates.json", "w", encoding="utf-8") as f:
    json.dump(explanation_templates, f, indent=2, ensure_ascii=False, default=json_default)

print("Saved:", ARTIFACT_DIR / "explanation_templates.json")


# ============================================================
# 11. SAVE MODEL CARD
# ============================================================

print_box("STEP 10 — Saving model_card.json")

model_card = {
    "model_name": "TurkQuish 5-class URL-only classifier",
    "model_version": "turkquish-5class-v1",
    "created_at": utc_now(),
    "task": "five_class_url_threat_classification",
    "classes": [
        "benign",
        "phishing",
        "malware",
        "scam",
        "other_malicious"
    ],
    "model_type": str(type(model_5class)),
    "model_variable_name": model_name,
    "model_classes_raw_order": model_classes_json,
    "model_classes_label_order": model_class_order_labels,
    "feature_schema_version": feature_schema["schema_version"],
    "n_features": len(feature_names),
    "url_only": True,
    "network_features_used": False,
    "forbidden_runtime_actions": [
        "DNS lookup",
        "WHOIS lookup",
        "HTML retrieval",
        "webpage screenshot",
        "third-party reputation query",
        "live webpage fetch"
    ],
    "risk_score": "1 - P(benign)",
    "threshold": float(tau),
    "graph_artifact_source": graph_source,
    "graph_protocol": (
        "Frozen training graph/projector; inference URLs are projected without updating graph."
    ),
    "limitations": [
        "Predictions are probabilistic and should not be treated as absolute truth.",
        "Five-class subtype prediction depends on the quality of the training labels and label harmonisation.",
        "The frozen graph/projector must match the graph-feature construction used during model training.",
        "Backend feature extraction must be identical to notebook feature extraction.",
        "If fallback token statistics were used instead of the original graph/projector, final deployment should replace them with the exact training graph/projector."
    ]
}

with open(ARTIFACT_DIR / "model_card.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False, default=json_default)

print("Saved:", ARTIFACT_DIR / "model_card.json")


# ============================================================
# 12. OPTIONAL: SAVE FEATURE IMPORTANCE
# ============================================================

print_box("STEP 11 — Saving optional feature_importance.json")

feature_importance = None

try:
    if hasattr(model_5class, "feature_importances_"):
        feature_importance = list(model_5class.feature_importances_)
    elif isinstance(model_5class, Pipeline):
        final_est = model_5class.steps[-1][1]
        if hasattr(final_est, "feature_importances_"):
            feature_importance = list(final_est.feature_importances_)
except Exception:
    feature_importance = None

if feature_importance is not None and len(feature_importance) == len(feature_names):
    importance_rows = [
        {
            "feature": feat,
            "importance": float(imp),
            "group": infer_feature_group(feat)
        }
        for feat, imp in zip(feature_names, feature_importance)
    ]
    importance_rows = sorted(importance_rows, key=lambda x: x["importance"], reverse=True)

    with open(ARTIFACT_DIR / "feature_importance.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "created_at": utc_now(),
                "source": "model.feature_importances_",
                "items": importance_rows
            },
            f,
            indent=2,
            ensure_ascii=False,
            default=json_default
        )

    print("Saved:", ARTIFACT_DIR / "feature_importance.json")
else:
    print("No compatible feature_importances_ found. Skipping feature_importance.json")


# ============================================================
# 13. VERIFY EXPORTED ARTIFACTS
# ============================================================

print_box("STEP 12 — Verifying exported artifacts")

required_files = [
    "model.joblib",
    "preprocessing_pipeline.joblib",
    "feature_schema.json",
    "label_encoder.json",
    "threshold.json",
    "frozen_token_graph.pkl",
    "explanation_templates.json",
    "model_card.json"
]

for filename in required_files:
    path = ARTIFACT_DIR / filename
    print(f"{filename:<35s}", "✅" if path.exists() else "❌")

loaded_model = joblib.load(ARTIFACT_DIR / "model.joblib")
loaded_preprocessor = joblib.load(ARTIFACT_DIR / "preprocessing_pipeline.joblib")

print("\nLoaded model type:", type(loaded_model))
print("Loaded model classes:", getattr(loaded_model, "classes_", None))

assert hasattr(loaded_model, "predict_proba"), "Loaded model does not support predict_proba()."
assert len(getattr(loaded_model, "classes_", [])) == 5, "Loaded model is not 5-class."

with open(ARTIFACT_DIR / "feature_schema.json", "r", encoding="utf-8") as f:
    loaded_schema = json.load(f)

assert loaded_schema["n_features"] == len(loaded_schema["features"])
assert loaded_schema["n_features"] == len(feature_names)

print("Feature count verified:", loaded_schema["n_features"])

# Try test prediction if X_train is available
if X_train_obj is not None and hasattr(X_train_obj, "columns"):
    print("\nTesting predict_proba on 3 training rows...")

    X_sample = X_train_obj[feature_names].head(3).copy()
    X_sample_proc = loaded_preprocessor.transform(X_sample)
    proba = loaded_model.predict_proba(X_sample_proc)

    print("Prediction probability shape:", proba.shape)
    print("Example probabilities:")
    print(proba)

    assert proba.shape[1] == 5, "predict_proba does not return 5 columns."
else:
    print("\nNo X_train sample available for prediction test. Artifact files were still verified.")


# ============================================================
# 14. ZIP ARTIFACTS
# ============================================================

print_box("STEP 13 — Creating ZIP package")

zip_path = shutil.make_archive(
    base_name=str(ARTIFACT_DIR),
    format="zip",
    root_dir=str(ARTIFACT_DIR)
)

print("Created ZIP:", zip_path)

print_box("EXPORT COMPLETE ✅")
print("Artifact folder:", ARTIFACT_DIR)
print("ZIP file:", zip_path)
print("\nCopy this folder into your FastAPI backend as:")
print("backend/app/artifacts/")


STEP 1 — Finding fitted 5-class model
5-class model candidates found:


RuntimeError: No fitted 5-class model found in notebook memory.
Run the 5-class training cell first, or set:
MANUAL_MODEL_VAR_NAME = 'your_model_variable_name'